
#Laboratory: Connect a LLM-based Reasoning Model with FrozenLake-v1
#================================================================
#Course: Intelligent Agents — Unit 3
#Topic: Connect a LLM-based Reasoning Model with FrozenLake-v1 / Implicit vs Explicit Reasoning
#Code generated by ChatGPT
#Adapted and Tested by Rolando Armas PhD / YachayTech
#Created at Oct 25
#Updated at Nov 26
#================================================================

In [1]:

# ================================================================
# 🧰 1. Environment Setup
# ================================================================
#!pip install -q gymnasium transformers torch numpy openai

import gymnasium as gym
from transformers import pipeline
import numpy as np
import json, time, os

# Optional OpenAI setup (comment out if not used)
#import openai
#openai.api_key = os.getenv("OPENAI_API_KEY", "YOUR_KEY_HERE")  # set your key or leave blank

# ================================================================
# 🌍 2. Initialize FrozenLake Environment
# ================================================================
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")

# Helper to convert numeric obs to human-readable text
def obs_to_text(env, obs):
    """Return a natural-language description of the current state."""
    desc = env.unwrapped.desc.astype(str)
    n = env.unwrapped.nrow
    r, c = divmod(int(obs), n)
    grid_lines = ["".join(row) for row in desc]
    grid_text = "\n".join(grid_lines)
    return f"Agent at row {r}, col {c}.\nMap:\n{grid_text}"



In [2]:
# ================================================================
# 💬 3. Define Prompt Templates
# ================================================================
implicit_template = """You are an expert navigator for frozen lakes.
Choose the best ACTION from LEFT, DOWN, RIGHT, UP.
Output only the action word.

Observation:
{obs_text}
"""

explicit_template = """You are an expert navigator for frozen lakes.
Explain your reasoning step-by-step, then output the final action.
Choose the best ACTION from LEFT, DOWN, RIGHT, UP.

Observation:
{obs_text}
"""

In [ ]:
implicit_template = """You are an expert navigator for frozen lakes.
Choose the best ACTION from LEFT, DOWN, RIGHT, UP. We need to go from S to G without falling in H. Could you provide the best route?
Output only the action word.

Observation:
{obs_text}
"""

explicit_template = """You are an expert navigator for frozen lakes.
We need to go from S to G without falling in H. Could you provide the best route? Explain your reasoning step-by-step, then output the final action.
Choose the best ACTION from LEFT, DOWN, RIGHT, UP.


Observation:
{obs_text}
"""

In [ ]:
implicit_template = """You are an expert navigator for frozen lakes.
We need to go from S to G without falling in H.
Choose the best ACTION from LEFT, DOWN, RIGHT, UP.

Consider the next navigation rules:
1. Stay on frozen tiles (F)
2. Avoid holes (H) at all costs.
3. When you are in the limits of the grid, you cannot go outside.
4. G is the goal.

Could you provide the best route?
Output only the action word.

Observation:
{obs_text}
"""

explicit_template = """You are an expert navigator for frozen lakes.
We need to go from S to G without falling in H.
Choose the best ACTION from LEFT, DOWN, RIGHT, UP.

Consider the next navigation rules:
1. Stay on frozen tiles (F)
2. Avoid holes (H) at all costs.
3. When you are in the limits of the grid, you cannot go outside.
4. G is the goal.


Could you provide the best route?
Explain your reasoning step-by-step, then output the final action.

Observation:
{obs_text}
"""

In [7]:
# ================================================================
# 🧠 4. Initialize LLM Interfaces
# ================================================================
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Option A — Local Hugging Face pipeline
# We load the model and tokenizer explicitly
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Bypassing the removed pipeline tasks by defining a direct generation wrapper
def generator(prompt, max_new_tokens=64, do_sample=False):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return [{"generated_text": generated_text}]

# Option B — OpenAI API (comment if not using)
#def ask_openai(prompt, model="gpt-4o-mini"):
#    """Query an OpenAI model and return its textual answer."""
#    try:
#        res = openai.ChatCompletion.create(
#            model=model,
#            messages=[{"role": "user", "content": prompt}],
#            temperature=0.0,
#            max_tokens=150
#        )
#        return res.choices[0].message.content
#    except Exception as e:
#        print("OpenAI error:", e)
#        return "ACTION: LEFT"  # fallback


# ================================================================
# ⚙️ 5. Action Parser
# ================================================================
def parse_action_from_text(text):
    """Extract one of {LEFT, DOWN, RIGHT, UP} from model output."""
    t = text.strip().upper()
    if "ACTION:" in t:
        t = t.split("ACTION:")[-1].strip()
    for token in ["LEFT","DOWN","RIGHT","UP"]:
        if token in t:
            return {"LEFT":0,"DOWN":1,"RIGHT":2,"UP":3}[token]
    return None  # unable to parse

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [8]:
# ================================================================
# 🔁 6. Main Loop: Environment ↔ LLM Reasoning ↔ Action
# ================================================================
def run_llm_agent(env, n_episodes=5, use_explicit=False, use_openai=False):
    """Run multiple episodes and record traces."""
    template = explicit_template if use_explicit else implicit_template
    results = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep)
        done = False
        total_reward = 0
        step = 0
        trace = []
        while not done and step < 10:
            obs_text = obs_to_text(env, obs)
            prompt = template.format(obs_text=obs_text)

            if use_openai:
                out = ask_openai(prompt)
            else:
                out = generator(prompt, max_new_tokens=64, do_sample=False)[0]['generated_text']

            #print(f"Prompt: {prompt}")
            #print(f"LLM output: {out}")
            action_id = parse_action_from_text(out)
            if action_id is None:
                action_id = env.action_space.sample()
                parse_fail = True
            else:
                parse_fail = False

            next_obs, reward, terminated, truncated, info = env.step(action_id)
            done = terminated or truncated
            total_reward += float(reward)
            trace.append({
                "step": step,
                "obs": obs_text,
                "prompt": prompt,
                "llm_text": out,
                "action": int(action_id),
                "parse_fail": parse_fail,
                "reward": float(reward)
            })
            obs = next_obs
            step += 1

        results.append({"episode": ep, "total_reward": total_reward, "trace": trace})
        print(f"Episode {ep} | Reward = {total_reward:.2f}")
        print(env.render())

    return results

In [9]:
# ================================================================
# ▶️ 7. Run Implicit vs Explicit Reasoning
# ================================================================
print("=== IMPLICIT REASONING ===")
results_implicit = run_llm_agent(env, n_episodes=3, use_explicit=False, use_openai=False)

print("=== EXPLICIT (CHAIN-OF-THOUGHT) REASONING ===")
results_explicit = run_llm_agent(env, n_episodes=3, use_explicit=True, use_openai=False)

# ================================================================
# 📊 8. Analyze and Compare Results
# ================================================================
def summarize(results):
    avg_reward = np.mean([r["total_reward"] for r in results])
    parse_errors = np.mean([
        np.mean([t["parse_fail"] for t in r["trace"]]) for r in results
    ])
    return {"avg_reward": round(avg_reward,2), "parse_error_rate": round(parse_errors,2)}

summary = {
    "implicit": summarize(results_implicit),
    "explicit": summarize(results_explicit)
}

print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))

# Save all traces
with open("frozenlake_llm_results.json","w") as f:
    json.dump({"implicit":results_implicit,"explicit":results_explicit}, f, indent=2)

print("\nResults saved to frozenlake_llm_results.json")

=== IMPLICIT REASONING ===
Episode 0 | Reward = 0.00
  (Right)
SFFF
FHFH
FFFH
HFFG

Episode 1 | Reward = 0.00
  (Right)
SFFF
FHFH
FFFH
HFFG

Episode 2 | Reward = 0.00
  (Right)
SFFF
FHFH
FFFH
HFFG

=== EXPLICIT (CHAIN-OF-THOUGHT) REASONING ===
Episode 0 | Reward = 0.00
  (Right)
SFFF
FHFH
FFFH
HFFG

Episode 1 | Reward = 0.00
  (Right)
SFFF
FHFH
FFFH
HFFG

Episode 2 | Reward = 0.00
  (Right)
SFFF
FHFH
FFFH
HFFG


=== SUMMARY ===
{
  "implicit": {
    "avg_reward": 0.0,
    "parse_error_rate": 0.0
  },
  "explicit": {
    "avg_reward": 0.0,
    "parse_error_rate": 0.0
  }
}

Results saved to frozenlake_llm_results.json
